# Task 3 — AQE Order-Effect Diagnostic: monthly_zone_demand

## Purpose

This is a diagnostic check, not a benchmark result, and it does not get appended to task3_benchmark_results.jsonl.

The monthly_zone_demand AQE notebook measured AQE disabled first, then AQE enabled second, and got 1.05x with no corresponding change in the physical plan. The same pattern was already confirmed on distance_by_weather, where reversing the order flipped the winner. This notebook repeats that same check on monthly_zone_demand.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

from data_intensive_computing.benchmark_helpers import (
    build_spark_session,
    register_delta_tables,
    timed_run,
    capture_explain,
    find_project_root,
)

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

spark = build_spark_session(app_name="Task3-AQE-OrderCheck-MonthlyZoneDemand")
register_delta_tables(spark, PROCESSED_DIR)

:: loading settings :: url = jar:file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rexchang/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rexchang/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-789e8c85-58ea-41aa-ab0d-5b538617ceee;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in c

Registered taxi_trips: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/taxi_trips
Registered weather: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/weather
Registered air_quality: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/air_quality
Registered taxi_zones: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/taxi_zones
Registered integrated_taxi_trips: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/integrated_taxi_trips


26/09/18 00:23:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Query (identical to the original AQE notebook)

In [2]:
QUERY_SQL = """
SELECT
    t.pickup_year,
    t.pickup_month,
    t.pickup_location_id,
    COALESCE(z.borough, 'Unknown') AS pickup_borough,
    COALESCE(z.zone, 'Unknown') AS pickup_zone,
    COUNT(t.trip_id) AS trip_count
FROM taxi_trips AS t
LEFT JOIN taxi_zones AS z
    ON t.pickup_location_id = z.location_id
GROUP BY
    t.pickup_year,
    t.pickup_month,
    t.pickup_location_id,
    COALESCE(z.borough, 'Unknown'),
    COALESCE(z.zone, 'Unknown')
ORDER BY t.pickup_year, t.pickup_month, trip_count DESC, t.pickup_location_id
"""

## Run 1 (measured first): AQE enabled

This is the reversed position. AQE enabled now goes first, so it loses whatever advantage came from running second in the original notebook.

In [3]:
spark.conf.set("spark.sql.adaptive.enabled", "true")

first_run_timing = timed_run(spark, "monthly_zone_demand_aqe_on_FIRST", QUERY_SQL, repetitions=5)
print("Run 1 — AQE enabled, measured first:", first_run_timing)

Run 1 — AQE enabled, measured first: {'query_name': 'monthly_zone_demand_aqe_on_FIRST', 'row_count': 793, 'repetitions': 5, 'all_times_seconds': [0.8055, 0.4687, 0.443, 0.4587, 0.5345], 'median_seconds': 0.4687, 'min_seconds': 0.443, 'max_seconds': 0.8055}


## Run 2 (measured second): AQE disabled

This is the reversed position. AQE disabled now goes second, so it gains whatever advantage the original notebook's AQE-enabled run had.

In [4]:
spark.conf.set("spark.sql.adaptive.enabled", "false")

second_run_timing = timed_run(spark, "monthly_zone_demand_aqe_off_SECOND", QUERY_SQL, repetitions=5)
print("Run 2 — AQE disabled, measured second:", second_run_timing)

Run 2 — AQE disabled, measured second: {'query_name': 'monthly_zone_demand_aqe_off_SECOND', 'row_count': 793, 'repetitions': 5, 'all_times_seconds': [0.3997, 0.4224, 0.3957, 0.4257, 0.3932], 'median_seconds': 0.3997, 'min_seconds': 0.3932, 'max_seconds': 0.4257}


## Sanity check: physical plan still shows no structural AQE effect

In [5]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
plan_check = capture_explain(spark.sql(QUERY_SQL), mode="formatted")
has_shuffle_read = "AQEShuffleRead" in plan_check
has_final_initial_split = "== Final Plan ==" in plan_check and "== Initial Plan ==" in plan_check
plans_differ = False
if has_final_initial_split:
    final_section = plan_check.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    initial_section = plan_check.split("== Initial Plan ==")[1]
    plans_differ = final_section.strip() != initial_section.strip()

print("AQEShuffleRead present:", has_shuffle_read)
print("Final Plan differs from Initial Plan:", plans_differ)

AQEShuffleRead present: False
Final Plan differs from Initial Plan: False


## Interpretation

Result: whichever configuration was measured second won, in both notebooks.

| | Measured first | Measured second | Which config "won" |
|---|---|---|---|
| Original notebook | AQE disabled: 0.4115s | AQE enabled: 0.3922s | AQE enabled (1.05x) |
| This notebook | AQE enabled: 0.4687s | AQE disabled: 0.3997s | AQE disabled (1.17x) |

The winner flipped along with the position in the sequence, not with the AQE setting itself. This confirms the original 1.05x was caused by measurement order, not by a real AQE effect. The plan check above backs this up too: still no AQEShuffleRead, still no difference between the Final and Initial Plan.

In [6]:
spark.stop()